-------------------------------------------------------------------------
*   PONTIFÍCIA UNIVERSIDADE CATÓLICA DE MINAS GERAIS
*   PROFESSOR: VICTOR SALES SILVA
*   ALUNO: DGEISON SERRÃO PEIXOTO
*   MATRÍCULA: **1366415**
*   ATIVIDADE: LEITURA DE ARQUIVO EM FORMATO CSV UTILIZANDO SPARK
-------------------------------------------------------------------------

# IMPORTAÇÃO DAS BIBLIOTECAS

In [25]:
%pip install pyspark
!pip install azure-storage-blob

In [26]:
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.types import TimestampType, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col
from pyspark.sql.functions import to_date
import xml.etree.ElementTree as ET
from azure.storage.blob import BlobServiceClient
from azure.storage.blob import BlobClient

In [27]:
spark = SparkSession.builder.getOrCreate()

# DEFINIÇÃO DAS VARIÁVEIS

In [28]:
storageaccount = 'stgaccount687878'
container = 'datalake-687878'
connection_string = 'DefaultEndpointsProtocol=https;AccountName=stgaccount687878;AccountKey=SUA_ACCOUNT_KEY_AQUI;EndpointSuffix=core.windows.net'
blob_file = 'bronze/DADOS_ALUNOS/DADOS_ALUNOS.xml'

In [29]:
def listar_arquivos_no_container(conn_string, container_name, prefixo=""):
  try:
      # 1. Conecta ao serviço de Blob
      blob_service_client = BlobServiceClient.from_connection_string(conn_string)

      # 2. Obtém o cliente para o contêiner
      container_client = blob_service_client.get_container_client(container_name)

      print(f"Buscando arquivos em '{container_name}' com o prefixo '{prefixo}'...")

      # 3. Lista os blobs (arquivos) que começam com o prefixo
      blob_list = container_client.list_blobs(name_starts_with=prefixo)

      lista_de_arquivos = []
      for blob in blob_list:
          print(f"  - {blob.name}")
          lista_de_arquivos.append(blob.name)

      if not lista_de_arquivos:
          print("\nNenhum arquivo encontrado neste caminho.")

      return lista_de_arquivos

  except Exception as e:
      print(f"Ocorreu um erro ao tentar listar os arquivos: {e}")
      return []


camada_para_verificar = 'bronze/'

print(f"--- Verificando o conteúdo da camada '{camada_para_verificar}' ---")
arquivos_encontrados = listar_arquivos_no_container(
    connection_string,
    container,
    prefixo=camada_para_verificar
)

print("\n--- Fim da verificação ---")

--- Verificando o conteúdo da camada 'bronze/' ---
Buscando arquivos em 'datalake-687878' com o prefixo 'bronze/'...
--- Verificando o conteúdo da camada 'bronze/' ---
Buscando arquivos em 'datalake-687878' com o prefixo 'bronze/'...
  - bronze/DADOS_ALUNOS/DADOS_ALUNOS.xml
  - bronze/DADOS_BANCARIOS/DADOS_BANCARIOS.xml
  - bronze/DADOS_ESTUDANTES/DADOS_ESTUDANTES.json
  - bronze/DADOS_EXAMES/DADOS_EXAMES.csv
  - bronze/DADOS_VOOS/DADOS_VOOS.parquet

--- Fim da verificação ---
  - bronze/DADOS_ALUNOS/DADOS_ALUNOS.xml
  - bronze/DADOS_BANCARIOS/DADOS_BANCARIOS.xml
  - bronze/DADOS_ESTUDANTES/DADOS_ESTUDANTES.json
  - bronze/DADOS_EXAMES/DADOS_EXAMES.csv
  - bronze/DADOS_VOOS/DADOS_VOOS.parquet

--- Fim da verificação ---


In [30]:
blob_file_na_nuvem = 'bronze/DADOS_EXAMES/DADOS_EXAMES.csv'
arquivo_local = 'DADOS_EXAMES.csv'

print(f"Baixando o arquivo '{blob_file_na_nuvem}' da nuvem...")
try:
    blob_client = BlobClient.from_connection_string(
        conn_str=connection_string,
        container_name=container,
        blob_name=blob_file_na_nuvem
    )
    with open(arquivo_local, "wb") as my_blob:
        blob_data = blob_client.download_blob()
        blob_data.readinto(my_blob)

    print(f"Arquivo salvo localmente como '{arquivo_local}' com sucesso!")

except Exception as e:
    print(f"Ocorreu um erro no download: {e}")
    arquivo_local = None




Baixando o arquivo 'bronze/DADOS_EXAMES/DADOS_EXAMES.csv' da nuvem...
Arquivo salvo localmente como 'DADOS_EXAMES.csv' com sucesso!


In [31]:
arquivo = 'DADOS_EXAMES.csv'

# LEITURA DO ARQUIVO CSV USANDO PANDAS

In [32]:
df = pd.read_csv(arquivo, sep=';')

# EXIBINDO UMA AMOSTRA DOS DADOS

In [33]:
df.head(5)

,IdUnidadeAtendimento,Cidade,Estado,Bairro,NumPedidoMedico,IdExame,Exame,SiglaExame,Material,SetorExame,QtdAmostrasColhidas,QtdExames,PrecoExame,DataPrevistaResultado,DataLiberacaoResultado
0,713,BELO HORIZONTE,MG,FUNCIONÁRIOS,PED-6217,3368,URINA - CULTURA,CTU,URINA,MICROBIOLOGIA,1,1,"9,1419",2022-03-29 22:43:00.000,2022-03-25 18:18:00.000
1,713,BELO HORIZONTE,MG,FUNCIONÁRIOS,PED-11486,3369,PARASITOLÓGICO,FP,FEZES,COPROLOGIA,1,1,"7,6343",2022-02-22 14:43:00.000,2022-02-22 09:01:43.000
2,713,BELO HORIZONTE,MG,FUNCIONÁRIOS,PED-11818,3368,URINA - CULTURA,CTU,URINA,MICROBIOLOGIA,1,1,"9,1419",2022-03-08 14:43:00.000,2022-02-28 05:52:40.000
3,713,BELO HORIZONTE,MG,FUNCIONÁRIOS,PED-11937,3368,URINA - CULTURA,CTU,URINA,MICROBIOLOGIA,1,1,"9,1419",2022-01-11 14:43:00.000,2022-01-05 07:20:58.000
4,713,BELO HORIZONTE,MG,FUNCIONÁRIOS,PED-12091,3370,GRAM - BACTERIOSCOPIA,GRAM-U,URINA,MICROBIOLOGIA,1,1,"1,7025",2023-01-05 20:43:00.000,2023-01-05 15:16:01.000


# EXIBINDO OS METADADOS (SCHEMA) DO ARQUIVO

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   IdUnidadeAtendimento    500000 non-null  int64 
 1   Cidade                  500000 non-null  object
 2   Estado                  500000 non-null  object
 3   Bairro                  500000 non-null  object
 4   NumPedidoMedico         500000 non-null  object
 5   IdExame                 500000 non-null  int64 
 6   Exame                   500000 non-null  object
 7   SiglaExame              500000 non-null  object
 8   Material                500000 non-null  object
 9   SetorExame              500000 non-null  object
 10  QtdAmostrasColhidas     500000 non-null  int64 
 11  QtdExames               500000 non-null  int64 
 12  PrecoExame              500000 non-null  object
 13  DataPrevistaResultado   500000 non-null  object
 14  DataLiberacaoResultado  500000 non-n

# AJUSTAR O SCHEMA DOS DADOS, SE NECESSÁRIO

In [35]:
df['Cidade'] = df['Cidade'].astype('string')
df['Estado'] = df['Estado'].astype('string')
df['Bairro'] = df['Bairro'].astype('string')
df['NumPedidoMedico'] = df['NumPedidoMedico'].astype('string')
df['Exame'] = df['Exame'].astype('string')
df['SiglaExame'] = df['SiglaExame'].astype('string')
df['Material'] = df['Material'].astype('string')
df['SetorExame'] = df['SetorExame'].astype('string')
df['PrecoExame'] = df['PrecoExame'].str.replace(',','.').astype('double')
df['DataPrevistaResultado'] = df['DataPrevistaResultado'].astype('datetime64[ns]')
df['DataLiberacaoResultado'] = df['DataLiberacaoResultado'].astype('datetime64[ns]')

In [37]:
df.head()

,IdUnidadeAtendimento,Cidade,Estado,Bairro,NumPedidoMedico,IdExame,Exame,SiglaExame,Material,SetorExame,QtdAmostrasColhidas,QtdExames,PrecoExame,DataPrevistaResultado,DataLiberacaoResultado
0,713,BELO HORIZONTE,MG,FUNCIONÁRIOS,PED-6217,3368,URINA - CULTURA,CTU,URINA,MICROBIOLOGIA,1,1,9.1419,2022-03-29 22:43:00,2022-03-25 18:18:00
1,713,BELO HORIZONTE,MG,FUNCIONÁRIOS,PED-11486,3369,PARASITOLÓGICO,FP,FEZES,COPROLOGIA,1,1,7.6343,2022-02-22 14:43:00,2022-02-22 09:01:43
2,713,BELO HORIZONTE,MG,FUNCIONÁRIOS,PED-11818,3368,URINA - CULTURA,CTU,URINA,MICROBIOLOGIA,1,1,9.1419,2022-03-08 14:43:00,2022-02-28 05:52:40
3,713,BELO HORIZONTE,MG,FUNCIONÁRIOS,PED-11937,3368,URINA - CULTURA,CTU,URINA,MICROBIOLOGIA,1,1,9.1419,2022-01-11 14:43:00,2022-01-05 07:20:58
4,713,BELO HORIZONTE,MG,FUNCIONÁRIOS,PED-12091,3370,GRAM - BACTERIOSCOPIA,GRAM-U,URINA,MICROBIOLOGIA,1,1,1.7025,2023-01-05 20:43:00,2023-01-05 15:16:01
